<a href="https://colab.research.google.com/github/MrImpeccable/UP/blob/main/Manuscript1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
"""
run_analysis.py
================================================================================
Frozen, single-execution analysis pipeline for:

  "Predicting compressive strength of SCM-blended mortar under sulphate
   exposure using ridge regression"

Run this ONCE. Every table and figure in the manuscript comes from this one
execution. Do not hand-edit any number afterwards.

    python run_analysis.py --data final_cleaned_sulfate_data.csv

Outputs land in ./run_output/ :
    audit_report.txt        data quality flags -- READ THIS FIRST
    table2_model_comparison.csv
    table3_coefficients.csv
    table4_cpa_predictions.csv
    loso_validation.csv     leave-one-source-out results
    equation.txt            raw-unit equation you can paste into Section 2.4
    figure3_actual_vs_pred.png
    figure4_cpa_timeseries.png
    figure5_strength_by_scm.png
    figure6_coefficients.png
    manifest.txt            seed, versions, alpha, row counts -- for Methods
================================================================================
"""

import argparse
import json
import platform
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV, LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import (KFold, LeaveOneGroupOut, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ==============================================================================
# CONFIGURATION -- edit this block to match your CSV, then leave it alone
# ==============================================================================

SEED = 42
TEST_SIZE = 0.20
CV_FOLDS = 5
ALPHA_GRID = np.logspace(-4, 4, 200)   # wide grid: this is the scaling fix

# Map your CSV's column headers to the names the script uses internally.
# Left = internal name, right = whatever your CSV actually calls it.
COLUMN_MAP = {
    "source":       "Source Title",
    "scm_type":     "SCM_Type",
    "scm_pct":      "SCM_%",
    "exposure":     "Exposure_Days",
    "sulphate":     "Sulfate_Conc_mol/L",
    "strength":     "Comp_Strength_MPa",
}

# Which SCM_Type value is the no-replacement control. All coefficients are
# interpreted relative to this. Blank/NaN cells are treated as control.
REFERENCE_SCM = "Control"

# Continuous predictors -- these get standardised.
CONTINUOUS = ["exposure", "sulphate"]

# Table 4 scenario
TABLE4_SCM = "CPA"
TABLE4_SULPHATE = 0.367
TABLE4_DAYS = [7, 14, 28, 56, 90]
TABLE4_EXTRAPOLATE = [180]      # reported separately as out-of-range

# Figure 5 projection range. Keep the ceiling near your data max unless you
# have the long-duration rows back in.
FIG5_MAX_DAYS = 90
FIG5_SULPHATE = 0.367

OUTDIR = Path("run_output")

# Expected dosage implied by each SCM_Type label, used only for the audit.
IMPLIED_DOSAGE = {
    "SF5": 5, "SF10": 10, "NS1": 1, "NS3": 3,
    "FA10": 10, "FA30": 30, "GBS10": 10, "GBS30": 30,
}

plt.rcParams.update({
    "figure.dpi": 200, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 9, "axes.grid": True, "grid.alpha": 0.3,
})


# ==============================================================================
# 1. LOAD AND NORMALISE
# ==============================================================================

def load(path):
    df = pd.read_csv(path) if str(path).lower().endswith(".csv") \
        else pd.read_excel(path)
    df.columns = [str(c).strip() for c in df.columns]

    missing = [src for src in COLUMN_MAP.values() if src not in df.columns]
    if missing:
        sys.exit(
            f"\nERROR: these columns are not in your file: {missing}\n"
            f"Your file has: {list(df.columns)}\n"
            f"Fix COLUMN_MAP at the top of this script.\n"
        )

    out = pd.DataFrame({k: df[v] for k, v in COLUMN_MAP.items()})
    out["scm_type"] = (out["scm_type"].astype(str).str.strip()
                       .replace({"nan": REFERENCE_SCM, "": REFERENCE_SCM,
                                 "None": REFERENCE_SCM}))
    for c in CONTINUOUS + ["strength"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out["source"] = out["source"].astype(str).str.strip()
    return out, df


# ==============================================================================
# 2. DATA AUDIT -- catches the inconsistencies a reviewer would find
# ==============================================================================

def audit(df, raw):
    L = ["DATA AUDIT", "=" * 78, ""]
    L.append(f"Rows as loaded: {len(df)}")
    L.append(f"Sources: {df['source'].nunique()} -> "
             f"{sorted(df['source'].unique())}")
    L.append(f"SCM types: {sorted(df['scm_type'].unique())}")
    L.append("")

    L.append("-- Missing values --")
    nulls = df.isna().sum()
    L += [f"  {k}: {v}" for k, v in nulls.items() if v] or ["  none"]
    L.append("")

    L.append("-- Exact duplicate rows --")
    d = df.duplicated().sum()
    L.append(f"  {d} duplicate row(s)"
             + (" <-- investigate, these inflate R^2" if d else ""))
    L.append("")

    L.append("-- SCM_% vs dosage implied by SCM_Type label --")
    bad = []
    for lbl, dose in IMPLIED_DOSAGE.items():
        sub = df[df["scm_type"] == lbl]
        if len(sub) and not np.allclose(sub["scm_pct"].dropna(), dose):
            vals = sorted(sub["scm_pct"].dropna().unique().tolist())
            bad.append(f"  {lbl}: label implies {dose}%, CSV has {vals}")
    L += bad or ["  consistent"]
    if bad:
        L.append("  <-- FIX THIS. SF5 rows recorded at 10% is a real error and")
        L.append("      it also means SCM_Type and SCM_% encode the same thing")
        L.append("      twice, which is a collinearity problem in itself.")
    L.append("")

    L.append("-- Control rows with non-zero sulphate concentration --")
    ctrl = df[df["scm_type"] == REFERENCE_SCM]
    odd = ctrl[ctrl["sulphate"] > 0]
    L.append(f"  {len(odd)} of {len(ctrl)} control rows"
             + (" <-- check whether these are unexposed controls" if len(odd)
                else ""))
    L.append("")

    L.append("-- Ranges --")
    for c in CONTINUOUS + ["strength"]:
        s = df[c].dropna()
        L.append(f"  {c:<10} n={len(s):<4} min={s.min():<10.4g} "
                 f"max={s.max():<10.4g} mean={s.mean():<10.4g} sd={s.std():.4g}")
    L.append("")

    L.append("-- Strength distribution (the bimodality problem) --")
    s = df["strength"].dropna()
    lo, hi = (s < 30).sum(), (s >= 30).sum()
    L.append(f"  below 30 MPa: {lo}   at/above 30 MPa: {hi}")
    L.append("  by source:")
    for src, g in df.groupby("source")["strength"]:
        L.append(f"    {src:<28} n={len(g):<4} "
                 f"range {g.min():.2f}-{g.max():.2f} MPa")
    L.append("  If sources occupy separate strength bands, a high R^2 mostly")
    L.append("  reflects the model identifying the source, not the chemistry.")
    L.append("  The leave-one-source-out result is the honest test.")
    L.append("")
    return "\n".join(L)


# ==============================================================================
# 3. DESIGN MATRIX
# ==============================================================================

def build_design(df):
    df = df.dropna(subset=CONTINUOUS + ["strength"]).reset_index(drop=True)

    dummies = pd.get_dummies(df["scm_type"], prefix="SCM", dtype=float)
    ref_col = f"SCM_{REFERENCE_SCM}"
    if ref_col in dummies.columns:
        dummies = dummies.drop(columns=[ref_col])
        ref_used = REFERENCE_SCM
    else:
        ref_used = dummies.columns[0].replace("SCM_", "")
        dummies = dummies.drop(columns=[dummies.columns[0]])

    X = pd.concat([df[CONTINUOUS], dummies], axis=1)
    y = df["strength"].astype(float)
    return X, y, df["source"], ref_used, list(dummies.columns)


def vif_table(X):
    """Multicollinearity check -- justifies ridge, or shows it isn't needed."""
    rows = []
    Xv = X.astype(float).values
    for i, name in enumerate(X.columns):
        others = np.delete(Xv, i, axis=1)
        if np.ptp(Xv[:, i]) == 0:
            rows.append({"feature": name, "VIF": np.nan}); continue
        r2 = LinearRegression().fit(others, Xv[:, i]).score(others, Xv[:, i])
        rows.append({"feature": name,
                     "VIF": np.inf if r2 >= 1 - 1e-12 else 1.0 / (1.0 - r2)})
    return pd.DataFrame(rows)


def make_pipe(model, X):
    """Standardise continuous predictors only; leave 0/1 dummies alone."""
    cont = [c for c in CONTINUOUS if c in X.columns]
    dums = [c for c in X.columns if c not in cont]
    pre = ColumnTransformer(
        [("scale", StandardScaler(), cont), ("pass", "passthrough", dums)],
        remainder="drop")
    return Pipeline([("pre", pre), ("model", model)]), cont + dums


# ==============================================================================
# 4. FIT, COMPARE, VALIDATE
# ==============================================================================

def metrics(y, p):
    return {"R2": r2_score(y, p),
            "MAE": mean_absolute_error(y, p),
            "RMSE": float(np.sqrt(mean_squared_error(y, p)))}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="path to your CSV or XLSX")
    ap.add_argument("--outdir", default=str(OUTDIR))
    # Simulate command-line arguments for Colab environment
    # Replace 'final_cleaned_sulfate_data.csv' with your actual data file name
    args = ap.parse_args(['--data', 'final_cleaned_sulfate_data_v2.csv']) # Use v2 data

    out = Path(args.outdir); out.mkdir(exist_ok=True)
    np.random.seed(SEED)

    df, raw = load(args.data)
    (out / "audit_report.txt").write_text(audit(df, raw))
    print(audit(df, raw))

    X, y, groups, ref_used, dummy_cols = build_design(df)
    print(f"\nModelling on {len(X)} rows, {X.shape[1]} predictors. "
          f"Reference category: {ref_used}\n")

    vif = vif_table(X)
    vif.to_csv(out / "vif.csv", index=False)
    print("Multicollinearity (VIF > 10 is the usual flag):")
    print(vif.round(2).to_string(index=False), "\n")

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED)

    # ---- alpha selection over a wide grid ------------------------------------
    ridge_cv, feat_order = make_pipe(
        RidgeCV(alphas=ALPHA_GRID, cv=KFold(CV_FOLDS, shuffle=True,
                                            random_state=SEED)), X)
    ridge_cv.fit(X_tr, y_tr)
    alpha = float(ridge_cv.named_steps["model"].alpha_)
    print(f"Selected alpha over grid 1e-4..1e4: {alpha:.4g}")
    if alpha < 0.1:
        print("  NOTE: alpha this small means ridge behaves almost exactly")
        print("  like OLS. Say so in the paper rather than claiming ridge")
        print("  overcame multicollinearity.\n")

    # ---- Table 2: model comparison ------------------------------------------
    candidates = {
        "OLS (baseline)":   LinearRegression(),
        "Ridge":            Ridge(alpha=alpha, random_state=None),
        "Lasso":            LassoCV(cv=CV_FOLDS, random_state=SEED,
                                    max_iter=100000),
        "Random Forest":    RandomForestRegressor(n_estimators=500,
                                                  random_state=SEED),
    }
    rows, fitted = [], {}
    cv = KFold(CV_FOLDS, shuffle=True, random_state=SEED)
    for name, est in candidates.items():
        pipe, _ = make_pipe(est, X)
        pipe.fit(X_tr, y_tr)
        fitted[name] = pipe
        tr = metrics(y_tr, pipe.predict(X_tr))
        te = metrics(y_te, pipe.predict(X_te))
        cvs = cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring="r2")
        rows.append({
            "Model": name,
            "R2_train": round(tr["R2"], 4),
            "R2_test": round(te["R2"], 4),
            "MAE_test": round(te["MAE"], 3),
            "RMSE_test": round(te["RMSE"], 3),
            "CV_R2_mean": round(float(cvs.mean()), 4),
            "CV_R2_sd": round(float(cvs.std()), 4),
        })
    table2 = pd.DataFrame(rows)
    table2.to_csv(out / "table2_model_comparison.csv", index=False)
    print("TABLE 2 -- model comparison\n" + table2.to_string(index=False), "\n")

    ridge = fitted["Ridge"]

    # ---- Leave-one-source-out: the validation that actually matters ----------
    loso_rows = []
    if groups.nunique() > 1:
        for tr_i, te_i in LeaveOneGroupOut().split(X, y, groups):
            held = groups.iloc[te_i].unique()[0]
            p, _ = make_pipe(Ridge(alpha=alpha), X)
            p.fit(X.iloc[tr_i], y.iloc[tr_i])
            m = metrics(y.iloc[te_i], p.predict(X.iloc[te_i]))
            loso_rows.append({"held_out_source": held, "n_test": len(te_i),
                              **{k: round(v, 4) for k, v in m.items()}})
    loso = pd.DataFrame(loso_rows)
    loso.to_csv(out / "loso_validation.csv", index=False)
    print("LEAVE-ONE-SOURCE-OUT (report this prominently)\n"
          + (loso.to_string(index=False) if len(loso) else "  only one source")
          + "\n")

    # ---- Table 3 + raw-unit equation ----------------------------------------
    scaler = ridge.named_steps["pre"].named_transformers_["scale"]
    coef = ridge.named_steps["model"].coef_
    intercept = float(ridge.named_steps["model"].intercept_)
    cont = [c for c in CONTINUOUS if c in X.columns]
    n_cont = len(cont)

    raw_coef, raw_intercept = {}, intercept
    for i, name in enumerate(cont):
        mu, sd = scaler.mean_[i], scaler.scale_[i]
        raw_coef[name] = coef[i] / sd
        raw_intercept -= coef[i] * mu / sd
    for j, name in enumerate(feat_order[n_cont:]):
        raw_coef[name] = coef[n_cont + j]

    table3 = pd.DataFrame([
        {"Variable": n,
         "Coef_standardised": round(float(coef[i]), 4),
         "Coef_raw_units": round(float(raw_coef[n]), 4),
         "Type": "continuous" if n in cont else "indicator"}
        for i, n in enumerate(feat_order)
    ])
    table3.loc[len(table3)] = {"Variable": "Intercept",
                               "Coef_standardised": round(intercept, 4),
                               "Coef_raw_units": round(raw_intercept, 4),
                               "Type": "intercept"}
    table3.to_csv(out / "table3_coefficients.csv", index=False)
    print("TABLE 3 -- coefficients\n" + table3.to_string(index=False), "\n")

    eq = [f"CS = {raw_intercept:.4f}"] + [
        f"  {'+' if raw_coef[n] >= 0 else '-'} {abs(raw_coef[n]):.4f} x {n}"
        for n in feat_order]
    eq_txt = (
        "Equation in RAW units (directly applicable, reproduces Table 4):\n\n"
        + "\n".join(eq)
        + f"\n\nReference category: {ref_used} (coefficient = 0 by construction)"
        + f"\nSelected alpha: {alpha:.6g}"
        + "\n\nStandardisation applied to continuous predictors before fitting:\n"
        + "\n".join(f"  {n}: mean={scaler.mean_[i]:.4f}, sd={scaler.scale_[i]:.4f}"
                    for i, n in enumerate(cont))
        + "\n\nState this standardisation explicitly in Section 2.4.\n"
    )
    (out / "equation.txt").write_text(eq_txt)
    print(eq_txt)

    # ---- Table 4 ------------------------------------------------------------
    def scenario(scm, days, sulph, pct=None):
        r = {c: 0.0 for c in X.columns}
        r["exposure"] = float(days)
        r["sulphate"] = float(sulph)
        if pct is None:
            sub = df[df["scm_type"] == scm]["scm_pct"]
            pct = float(sub.median()) if len(sub) else 0.0
        r["scm_pct"] = pct
        col = f"SCM_{scm}"
        if col in X.columns:
            r[col] = 1.0
        return pd.DataFrame([r])[X.columns]

    t4 = []
    obs = df[(df["scm_type"] == TABLE4_SCM) &
             np.isclose(df["sulphate"], TABLE4_SULPHATE, atol=1e-3)]
    for d in TABLE4_DAYS:
        a = obs[obs["exposure"] == d]["strength"]
        t4.append({
            "Exposure_days": d,
            "Actual_MPa": round(float(a.mean()), 2) if len(a) else np.nan,
            "n_obs": len(a),
            "Predicted_MPa": round(float(
                ridge.predict(scenario(TABLE4_SCM, d, TABLE4_SULPHATE))[0]), 2),
            "In_training_range": "yes",
        })
    for d in TABLE4_EXTRAPOLATE:
        t4.append({
            "Exposure_days": d, "Actual_MPa": np.nan, "n_obs": 0,
            "Predicted_MPa": round(float(
                ridge.predict(scenario(TABLE4_SCM, d, TABLE4_SULPHATE))[0]), 2),
            "In_training_range": "NO -- extrapolation",
        })
    table4 = pd.DataFrame(t4)
    table4.to_csv(out / "table4_cpa_predictions.csv", index=False)
    print(f"TABLE 4 -- {TABLE4_SCM} at {TABLE4_SULPHATE} mol/L\n"
          + table4.to_string(index=False), "\n")

    # ==========================================================================
    # FIGURES -- all from this same fitted model
    # ==========================================================================
    p_te = ridge.predict(X_te)

    # Figure 3
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(y_te, p_te, s=45, alpha=.8, edgecolor="k", linewidth=.5,
               zorder=3, label="Test set")
    lim = [min(y.min(), p_te.min()) - 3, max(y.max(), p_te.max()) + 3]
    ax.plot(lim, lim, "r--", lw=1.2, label="Actual = predicted")
    ax.set(xlim=lim, ylim=lim,
           xlabel="Actual compressive strength (MPa)",
           ylabel="Predicted compressive strength (MPa)",
           title=f"Actual vs predicted (ridge, $\\alpha$={alpha:.3g})")
    ax.legend(); fig.savefig(out / "figure3_actual_vs_pred.png"); plt.close(fig)

    # Figure 4
    fig, ax = plt.subplots(figsize=(6, 4))
    if len(obs):
        ax.scatter(obs["exposure"], obs["strength"], marker="x", c="tab:blue",
                   s=60, label="Measured", zorder=3)
    grid = np.linspace(min(TABLE4_DAYS), max(TABLE4_DAYS), 200)
    ax.plot(grid, [ridge.predict(scenario(TABLE4_SCM, d, TABLE4_SULPHATE))[0]
                   for d in grid], "r-", lw=1.6, label="Ridge prediction")
    if TABLE4_EXTRAPOLATE:
        eg = np.linspace(max(TABLE4_DAYS), max(TABLE4_EXTRAPOLATE), 100)
        ax.plot(eg, [ridge.predict(scenario(TABLE4_SCM, d, TABLE4_SULPHATE))[0]
                     for d in eg], "r:", lw=1.6, label="Extrapolated")
        ax.axvline(max(TABLE4_DAYS), color="grey", ls="--", lw=.8)
    ax.set(xlabel="Exposure (days)", ylabel="Compressive strength (MPa)",
           title=f"Actual vs predicted, {TABLE4_SCM} "
                 f"({TABLE4_SULPHATE} mol/L sulphate)")
    ax.legend(); fig.savefig(out / "figure4_cpa_timeseries.png"); plt.close(fig)

    # Figure 5
    fig, ax = plt.subplots(figsize=(6.5, 4))
    days = np.linspace(df["exposure"].min(), FIG5_MAX_DAYS, 100)
    for scm in sorted(df["scm_type"].unique()):
        ax.plot(days, [ridge.predict(scenario(scm, d, FIG5_SULPHATE))[0]
                       for d in days], lw=1.3, label=scm)
    ax.set(xlabel="Exposure (days)",
           ylabel="Predicted compressive strength (MPa)",
           title=f"Predicted strength by SCM type ({FIG5_SULPHATE} mol/L)")
    ax.legend(fontsize=7, ncol=2, loc="best")
    fig.savefig(out / "figure5_strength_by_scm.png"); plt.close(fig)

    # Figure 6 -- standardised coefficients, so bars are comparable
    fig, ax = plt.subplots(figsize=(6, 4.5))
    cd = table3[table3["Type"] != "intercept"].copy()
    cd = cd.sort_values("Coef_standardised")
    ax.barh(cd["Variable"], cd["Coef_standardised"],
            color=["tab:red" if v < 0 else "tab:blue"
                   for v in cd["Coef_standardised"]])
    ax.axvline(0, color="k", lw=.8)
    ax.set(xlabel="Standardised coefficient (MPa per SD, or vs reference)",
           title="Ridge coefficients")
    fig.savefig(out / "figure6_coefficients.png"); plt.close(fig)

    # ---- manifest -----------------------------------------------------------
    (out / "manifest.txt").write_text("\n".join([
        "FROZEN RUN MANIFEST",
        "=" * 50,
        f"Timestamp:        {datetime.now().isoformat(timespec='seconds')}",
        f"Data file:        {args.data}",
        f"Rows modelled:    {len(X)}",
        f"Predictors:       {X.shape[1]}",
        f"Reference cat.:   {ref_used}",
        f"Random seed:      {SEED}",
        f"Train/test:       {1-TEST_SIZE:.0%}/{TEST_SIZE:.0%}",
        f"CV folds:         {CV_FOLDS}",
        f"Alpha grid:       1e-4 to 1e4, {len(ALPHA_GRID)} points",
        f"Selected alpha:   {alpha:.6g}",
        "",
        f"Python:           {platform.python_version()}",
        f"numpy:            {np.__version__}",
        f"pandas:           {pd.__version__}",
        f"scikit-learn:     {sklearn.__version__}",
        f"matplotlib:       {matplotlib.__version__}",
        "",
        "Cite these versions in Section 2.4. Replace the 'Python 3.0' claim.",
    ]))

    print(f"Done. Everything written to {out.resolve()}")
    print("Read audit_report.txt before you touch the manuscript.")


if __name__ == "__main__":
    main()

DATA AUDIT

Rows as loaded: 94
Sources: 3 -> ['Abdulwahab+et+al', 'Diego+et+al', 'Wang+et+al']
SCM types: ['CPA', 'Control', 'FA10', 'FA30', 'GBS10', 'GBS30', 'NS1', 'NS3', 'RHA', 'SF10', 'SF5']

-- Missing values --
  none

-- Exact duplicate rows --
  6 duplicate row(s) <-- investigate, these inflate R^2

-- SCM_% vs dosage implied by SCM_Type label --
  consistent

-- Control rows with non-zero sulphate concentration --
  4 of 4 control rows <-- check whether these are unexposed controls

-- Ranges --
  exposure   n=94   min=0          max=500        mean=96.37      sd=142.2
  sulphate   n=94   min=0.352      max=0.704      mean=0.447      sd=0.1282
  strength   n=94   min=6.12       max=66         mean=36.39      sd=22.65

-- Strength distribution (the bimodality problem) --
  below 30 MPa: 40   at/above 30 MPa: 54
  by source:
    Abdulwahab+et+al             n=40   range 6.12-21.50 MPa
    Diego+et+al                  n=18   range 43.00-64.00 MPa
    Wang+et+al                   

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

output_dir = Path("run_output")
zip_filename = "run_output.zip"

if output_dir.exists() and output_dir.is_dir():
    shutil.make_archive(str(output_dir), 'zip', output_dir)
    files.download(zip_filename)
    print(f"'{zip_filename}' downloaded successfully.")
else:
    print(f"Error: The directory '{output_dir}' does not exist. Please ensure the analysis script has been run successfully to generate outputs.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'run_output.zip' downloaded successfully.


In [ ]:
from pathlib import Path
from google.colab import files

output_dir = Path("run_output")
summary_file_name = "summary_report.txt"

# List of text files to combine
text_files_to_combine = [
    output_dir / "audit_report.txt",
    output_dir / "equation.txt",
    output_dir / "manifest.txt",
]

combined_content = []
for f_path in text_files_to_combine:
    if f_path.exists():
        combined_content.append(f"--- Content of {f_path.name} ---\n")
        combined_content.append(f_path.read_text())
        combined_content.append("\n\n")
    else:
        print(f"Warning: {f_path.name} not found and will be skipped.")

if combined_content:
    with open(summary_file_name, "w") as f:
        f.write("".join(combined_content))
    files.download(summary_file_name)
    print(f"'{summary_file_name}' downloaded successfully, containing combined text reports.")
else:
    print("No text reports found to combine.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'summary_report.txt' downloaded successfully, containing combined text reports.


In [ ]:
#!/usr/bin/env python3
"""
clean_data.py
================================================================================
Cleans final_cleaned_sulfate_data.csv before the analysis rerun.

Does three things:
  1. Reports what is actually in the file (category counts, RHA count, SF rows)
  2. Removes exact duplicate rows
  3. Applies label fixes you configure below

Run it TWICE:

  Pass 1 -- report only, changes nothing:
      python clean_data.py --data final_cleaned_sulfate_data.csv --report-only

      Read the output. Decide what the bare "SF" rows should become.

  Pass 2 -- apply fixes after you have set SF_MERGE below:
      python clean_data.py --data final_cleaned_sulfate_data.csv

      Writes final_cleaned_sulfate_data_v2.csv and cleaning_log.txt
================================================================================
"""

import argparse
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# ==============================================================================
# CONFIGURATION
# ==============================================================================

COL_SOURCE   = "Source Title"
COL_SCM_TYPE = "SCM_Type"
COL_SCM_PCT  = "SCM_%"
COL_EXPOSURE = "Exposure_Days"
COL_SULPHATE = "Sulfate_Conc_mol/L"
COL_STRENGTH = "Comp_Strength_MPa"

CONTROL_LABEL = "Control"

# Dosage each label implies. SCM_% is rewritten from this, because the label
# is the more specific record. Verify against your source papers first.
IMPLIED_DOSAGE = {
    "SF5": 5, "SF10": 10,
    "NS1": 1, "NS3": 3,
    "FA10": 10, "FA30": 30,
    "GBS10": 10, "GBS30": 30,
    CONTROL_LABEL: 0,
}

# Labels with no dosage in the name. Leave SCM_% as recorded in the CSV.
DOSAGE_FROM_CSV = ["CPA", "RHA"]

# What the bare "SF" rows should become. Leave as None until pass 1 tells you
# what those rows contain. Then set one of:
#     SF_MERGE = "SF5"           merge all SF rows into SF5
#     SF_MERGE = "SF10"          merge all SF rows into SF10
#     SF_MERGE = "by_pct"        use each row's SCM_% (5 -> SF5, 10 -> SF10)
#     SF_MERGE = "drop"          remove the SF rows entirely
SF_MERGE = "SF10"

# Categories with fewer than this many rows get flagged as too thin to
# interpret. Their coefficients are noise.
MIN_ROWS_PER_CATEGORY = 6

# Rewrite sulphate concentration to 0 for unexposed control rows.
# Set True only if your control rows are genuinely water-cured, not immersed.
ZERO_SULPHATE_FOR_CONTROL = False


# ==============================================================================

def load(path):
    df = pd.read_csv(path) if str(path).lower().endswith(".csv") \
        else pd.read_excel(path)
    df.columns = [str(c).strip() for c in df.columns]
    need = [COL_SOURCE, COL_SCM_TYPE, COL_SCM_PCT, COL_EXPOSURE,
            COL_SULPHATE, COL_STRENGTH]
    missing = [c for c in need if c not in df.columns]
    if missing:
        sys.exit(f"ERROR: missing columns {missing}\nFile has: {list(df.columns)}")
    df[COL_SCM_TYPE] = (df[COL_SCM_TYPE].astype(str).str.strip()
                        .replace({"nan": CONTROL_LABEL, "": CONTROL_LABEL,
                                  "None": CONTROL_LABEL}))
    for c in [COL_SCM_PCT, COL_EXPOSURE, COL_SULPHATE, COL_STRENGTH]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def report(df, log):
    def P(s=""):
        print(s); log.append(s)

    P("=" * 78)
    P("DATASET REPORT")
    P("=" * 78)
    P(f"Rows: {len(df)}")
    P("")

    P("-- Rows per SCM category (this answers the RHA question) --")
    P(f"{'Category':<12}{'n':>5}{'sources':>9}   {'strength range':<22}"
      f"{'SCM_% values':<16}{'exposure days'}")
    P("-" * 78)
    thin = []
    for scm, g in df.groupby(COL_SCM_TYPE):
        pcts = sorted(g[COL_SCM_PCT].dropna().unique().tolist())
        days = sorted(g[COL_EXPOSURE].dropna().unique().tolist())
        rng = f"{g[COL_STRENGTH].min():.2f}-{g[COL_STRENGTH].max():.2f} MPa"
        flag = "  <-- THIN" if len(g) < MIN_ROWS_PER_CATEGORY else ""
        if len(g) < MIN_ROWS_PER_CATEGORY:
            thin.append((scm, len(g)))
        P(f"{scm:<12}{len(g):>5}{g[COL_SOURCE].nunique():>9}   "
          f"{rng:<22}{str(pcts):<16}{days}{flag}")
    P("")
    if thin:
        P("  THIN CATEGORIES -- their coefficients are noise, not findings:")
        for scm, n in thin:
            P(f"    {scm}: {n} rows")
        P("  Either merge them into a broader class (e.g. all agro-ashes")
        P("  together) or exclude them and say so in the Limitations.")
        P("")

    P("-- Bare 'SF' rows (decide what these become) --")
    sf = df[df[COL_SCM_TYPE] == "SF"]
    if len(sf):
        P(f"  {len(sf)} rows:")
        cols = [COL_SOURCE, COL_SCM_PCT, COL_EXPOSURE, COL_SULPHATE,
                COL_STRENGTH]
        for line in sf[cols].to_string(index=False).split("\n"):
            P("    " + line)
        P("  Set SF_MERGE at the top of this script, then rerun.")
    else:
        P("  none present")
    P("")

    P("-- Exact duplicate rows --")
    dup = df[df.duplicated(keep=False)]
    P(f"  {df.duplicated().sum()} removable duplicate(s), "
      f"{len(dup)} rows involved")
    if len(dup):
        for line in dup.sort_values(list(df.columns)).to_string(
                index=False).split("\n")[:20]:
            P("    " + line)
    P("")

    P("-- SCM_% disagreeing with the label --")
    bad = 0
    for lbl, dose in IMPLIED_DOSAGE.items():
        g = df[df[COL_SCM_TYPE] == lbl]
        if len(g) and not np.allclose(g[COL_SCM_PCT].dropna(), dose):
            vals = sorted(g[COL_SCM_PCT].dropna().unique().tolist())
            P(f"  {lbl}: label says {dose}%, CSV has {vals} ({len(g)} rows)")
            bad += 1
    P("  all consistent" if not bad else
      "  These will be rewritten from the label.")
    P("")

    P("-- Control rows --")
    ctrl = df[df[COL_SCM_TYPE] == CONTROL_LABEL]
    P(f"  {len(ctrl)} control rows, "
      f"{(ctrl[COL_SULPHATE] > 0).sum()} with non-zero sulphate")
    P(f"  ZERO_SULPHATE_FOR_CONTROL is currently {ZERO_SULPHATE_FOR_CONTROL}")
    P("")

    P("-- Sulphate concentration by source (the VIF=744 problem) --")
    for src, g in df.groupby(COL_SOURCE):
        P(f"  {src:<24} {sorted(g[COL_SULPHATE].dropna().unique().tolist())}")
    P("  If each source used one fixed concentration, sulphate is a proxy for")
    P("  study identity and its coefficient cannot be read as a chemical")
    P("  effect. Say this in the Discussion regardless of what you drop.")
    P("")

    P("-- Exposure coverage --")
    ct = pd.crosstab(df[COL_SOURCE], df[COL_EXPOSURE])
    for line in ct.to_string().split("\n"):
        P("  " + line)
    P("")
    return log


def clean(df, log):
    def P(s=""):
        print(s); log.append(s)

    P("=" * 78)
    P("APPLYING FIXES")
    P("=" * 78)
    n0 = len(df)

    # 1 -- duplicates
    df = df.drop_duplicates().reset_index(drop=True)
    P(f"1. Duplicates removed: {n0 - len(df)}  ({n0} -> {len(df)} rows)")

    # 2 -- SF merge
    n_sf = (df[COL_SCM_TYPE] == "SF").sum()
    if n_sf:
        if SF_MERGE is None:
            sys.exit(f"\nSTOP: {n_sf} bare 'SF' rows and SF_MERGE is None.\n"
                     f"Run with --report-only, read the SF rows, then set\n"
                     f"SF_MERGE at the top of this script.\n")
        if SF_MERGE == "drop":
            df = df[df[COL_SCM_TYPE] != "SF"].reset_index(drop=True)
            P(f"2. SF rows dropped: {n_sf}")
        elif SF_MERGE == "by_pct":
            m = df[COL_SCM_TYPE] == "SF"
            df.loc[m, COL_SCM_TYPE] = df.loc[m, COL_SCM_PCT].map(
                lambda p: "SF5" if p == 5 else ("SF10" if p == 10 else "SF"))
            left = (df[COL_SCM_TYPE] == "SF").sum()
            P(f"2. SF mapped by SCM_%; {n_sf - left} reassigned, {left} unmapped")
            if left:
                P("   <-- unmapped SF rows remain. Their SCM_% is neither 5 "
                  "nor 10.")
        else:
            df.loc[df[COL_SCM_TYPE] == "SF", COL_SCM_TYPE] = SF_MERGE
            P(f"2. SF -> {SF_MERGE}: {n_sf} rows")
    else:
        P("2. No bare SF rows")

    # 3 -- dosage from label
    fixed = 0
    for lbl, dose in IMPLIED_DOSAGE.items():
        m = df[COL_SCM_TYPE] == lbl
        if m.any():
            n = int((df.loc[m, COL_SCM_PCT] != dose).sum())
            if n:
                df.loc[m, COL_SCM_PCT] = dose
                P(f"   {lbl}: {n} rows rewritten to {dose}%")
                fixed += n
    P(f"3. SCM_% rewritten from label: {fixed} rows")
    for lbl in DOSAGE_FROM_CSV:
        if (df[COL_SCM_TYPE] == lbl).any():
            vals = sorted(df.loc[df[COL_SCM_TYPE] == lbl,
                                 COL_SCM_PCT].dropna().unique().tolist())
            P(f"   {lbl}: SCM_% left as recorded {vals}")

    # 4 -- control sulphate
    if ZERO_SULPHATE_FOR_CONTROL:
        m = df[COL_SCM_TYPE] == CONTROL_LABEL
        n = int((df.loc[m, COL_SULPHATE] != 0).sum())
        df.loc[m, COL_SULPHATE] = 0.0
        P(f"4. Control sulphate zeroed: {n} rows")
    else:
        P("4. Control sulphate left unchanged")

    # duplicates can reappear once labels are normalised
    n1 = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    if n1 != len(df):
        P(f"5. Duplicates created by relabelling, removed: {n1 - len(df)}")

    P("")
    P("-- Final category counts --")
    for scm, g in df.groupby(COL_SCM_TYPE):
        P(f"   {scm:<12}{len(g):>4} rows"
          + ("   <-- THIN" if len(g) < MIN_ROWS_PER_CATEGORY else ""))
    P(f"\nFinal: {len(df)} rows ({n0 - len(df)} removed overall)")
    return df


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", default=None)
    ap.add_argument("--report-only", action="store_true")
    # Simulate command-line arguments for Colab environment
    # Use 'final_cleaned_sulfate_data.csv' as the default data file
    args = ap.parse_args(['--data', 'final_cleaned_sulfate_data.csv'])

    df = load(args.data)
    log = []
    report(df, log)

    if args.report_only:
        Path("cleaning_report.txt").write_text("\n".join(log))
        print("\nReport only. Nothing written to the dataset.")
        print("Set SF_MERGE, then rerun without --report-only.")
        return

    df = clean(df, log)
    out = args.out or str(Path(args.data).with_name(
        Path(args.data).stem + "_v2.csv"))
    df.to_csv(out, index=False)
    Path("cleaning_log.txt").write_text("\n".join(log))
    print(f"\nWritten: {out}")
    print("Log:     cleaning_log.txt")
    print("\nNext: edit run_analysis.py, change")
    print('  CONTINUOUS = ["scm_pct", "exposure", "sulphate"]')
    print("to")
    print('  CONTINUOUS = ["exposure", "sulphate"]')
    print(f"then run:  python run_analysis.py --data {out}")


if __name__ == "__main__":
    main()

DATASET REPORT
Rows: 96

-- Rows per SCM category (this answers the RHA question) --
Category        n  sources   strength range        SCM_% values    exposure days
------------------------------------------------------------------------------
CPA            40        1   6.12-21.50 MPa        [20]            [7, 14, 28, 56, 90]
Control         4        1   48.00-52.00 MPa       [0]             [7, 28, 180, 500]  <-- THIN
FA10            4        1   52.00-57.00 MPa       [10]            [7, 28, 180, 500]  <-- THIN
FA30            4        1   47.00-52.00 MPa       [10]            [7, 28, 180, 500]  <-- THIN
GBS10           4        1   55.00-60.00 MPa       [10]            [7, 28, 180, 500]  <-- THIN
GBS30           4        1   49.00-54.00 MPa       [10]            [7, 28, 180, 500]  <-- THIN
NS1             4        1   50.00-56.00 MPa       [10]            [7, 28, 180, 500]  <-- THIN
NS3             4        1   52.00-58.00 MPa       [10]            [7, 28, 180, 500]  <-- THIN
RHA

In [ ]:
import os
import zipfile
from google.colab import files

def zip_directory(path, zip_file_handle):
    for root, dirs, files_in_dir in os.walk(path):
        for file in files_in_dir:
            zip_file_handle.write(os.path.join(root, file), os.path.relpath(os.path.join(root, file), os.path.join(path, '..')))

zip_filename = 'all_files.zip'
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zip_directory('.', zipf)

files.download(zip_filename)
print(f"All files zipped into '{zip_filename}' and ready for download.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All files zipped into 'all_files.zip' and ready for download.


In [ ]:
#!/usr/bin/env python3
"""
make_figures.py
================================================================================
Generates all six manuscript figures for

  "Between-study variance inflates apparent accuracy in literature-compiled
   machine learning models of sulphate resistance in SCM-blended cement"

    python make_figures.py --data final_cleaned_sulfate_data_v2.csv

Outputs to ./figures/ :
    Fig1.tif  dataset composition by source and SCM category
    Fig2.tif  predicted vs measured, random test partition
    Fig3.tif  standardised ridge coefficients
    Fig4.tif  R2 under random partitioning vs leave-one-source-out
    Fig5.tif  strength distribution by source
    Fig6.tif  CPA measured vs predicted over exposure

Journal requirements applied throughout:
  - No titles inside the figure. Titles are captions in the manuscript.
  - Arial lettering, 8-12 pt, minimal size variation within a figure.
  - TIFF, 300 DPI (halftone/combination). Set --eps for vector output instead.
  - Line widths at or above 0.3 pt.
  - Patterns and markers as well as colour, for colour-blind readers.
  - Figure parts labelled with lowercase letters where a figure has parts.

Everything is fitted with the same seed, alpha grid and preparation as
run_analysis.py, so the numbers here match Tables 3-7.
================================================================================
"""

import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.model_selection import KFold, LeaveOneGroupOut, train_test_split
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ==============================================================================
# CONFIGURATION -- must match run_analysis.py
# ==============================================================================

SEED = 42
TEST_SIZE = 0.20
CV_FOLDS = 5
ALPHA_GRID = np.logspace(-4, 4, 200)

COLUMN_MAP = {
    "source":   "Source Title",
    "scm_type": "SCM_Type",
    "scm_pct":  "SCM_%",
    "exposure": "Exposure_Days",
    "sulphate": "Sulfate_Conc_mol/L",
    "strength": "Comp_Strength_MPa",
}
REFERENCE_SCM = "Control"
CONTINUOUS = ["exposure", "sulphate"]      # scm_pct dropped (see Section 2.3)

# Display names for the three sources. Replace once citations are resolved.
SOURCE_LABELS = {
    "Abdulwahab+et+al": "Source A",
    "Diego+et+al":      "Source B",
    "Wang+et+al":       "Source C",
}
SOURCE_ORDER = ["Source A", "Source B", "Source C"]

# Figure 6 scenario
F6_SCM = "CPA"
F6_SULPHATE = 0.367
F6_OBS_MAX = 90        # end of observed range
F6_EXTRAP_MAX = 180    # dotted beyond this point

OUTDIR = Path("figures")

# ==============================================================================
# STYLE
# ==============================================================================

def set_style(fmt):
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": 9,            # body lettering
        "axes.titlesize": 9,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "axes.linewidth": 0.6,     # ~0.4 pt, above the 0.3 pt floor
        "grid.linewidth": 0.5,
        "lines.linewidth": 1.2,
        "xtick.major.width": 0.6,
        "ytick.major.width": 0.6,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": "-",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
    })

# Greyscale-safe palette. Distinguishable in colour and when printed mono.
C_DARK = "#1f3b5c"
C_MID = "#5b8db8"
C_LIGHT = "#a8c4dc"
C_NEG = "#8c2d2d"
C_POS = "#2f5f8f"
C_LINE = "#b03030"
SOURCE_FILL = {"Source A": C_DARK, "Source B": C_MID, "Source C": C_LIGHT}
SOURCE_HATCH = {"Source A": "///", "Source B": "...", "Source C": ""}
SOURCE_MARK = {"Source A": "o", "Source B": "s", "Source C": "^"}

# Single-column width 78 mm, double-column 119 mm, per journal guidance.
W1 = 78 / 25.4
W2 = 119 / 25.4


def save(fig, name, fmt, outdir):
    outdir.mkdir(exist_ok=True)
    path = outdir / f"{name}.{fmt}"
    if fmt == "tif":
        fig.savefig(path, format="tiff", pil_kwargs={"compression": "tiff_lzw"})
    else:
        fig.savefig(path, format=fmt)
    plt.close(fig)
    print(f"  wrote {path}")


# ==============================================================================
# DATA AND MODEL
# ==============================================================================

def load(path):
    df = pd.read_csv(path) if str(path).lower().endswith(".csv") else pd.read_excel(path)
    df.columns = [str(c).strip() for c in df.columns]
    out = pd.DataFrame({k: df[v] for k, v in COLUMN_MAP.items()})
    out["scm_type"] = (out["scm_type"].astype(str).str.strip()
                       .replace({"nan": REFERENCE_SCM, "": REFERENCE_SCM,
                                 "None": REFERENCE_SCM}))
    for c in ["scm_pct", "exposure", "sulphate", "strength"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out["source"] = (out["source"].astype(str).str.strip()
                     .map(lambda s: SOURCE_LABELS.get(s, s)))
    return out.dropna(subset=CONTINUOUS + ["strength"]).reset_index(drop=True)


def build_design(df):
    dummies = pd.get_dummies(df["scm_type"], prefix="SCM", dtype=float)
    ref = f"SCM_{REFERENCE_SCM}"
    dummies = dummies.drop(columns=[ref]) if ref in dummies else dummies.iloc[:, 1:]
    X = pd.concat([df[CONTINUOUS], dummies], axis=1)
    return X, df["strength"].astype(float)


def make_pipe(model, X):
    cont = [c for c in CONTINUOUS if c in X.columns]
    dums = [c for c in X.columns if c not in cont]
    pre = ColumnTransformer([("scale", StandardScaler(), cont),
                             ("pass", "passthrough", dums)])
    return Pipeline([("pre", pre), ("model", model)]), cont + dums


def fit_everything(df):
    X, y = build_design(df)
    groups = df["source"]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED)

    cv = KFold(CV_FOLDS, shuffle=True, random_state=SEED)
    rcv, feat_order = make_pipe(RidgeCV(alphas=ALPHA_GRID, cv=cv), X)
    rcv.fit(X_tr, y_tr)
    alpha = float(rcv.named_steps["model"].alpha_)

    ridge, _ = make_pipe(Ridge(alpha=alpha), X)
    ridge.fit(X_tr, y_tr)

    loso = []
    for tr_i, te_i in LeaveOneGroupOut().split(X, y, groups):
        held = groups.iloc[te_i].unique()[0]
        p, _ = make_pipe(Ridge(alpha=alpha), X)
        p.fit(X.iloc[tr_i], y.iloc[tr_i])
        loso.append({"source": held,
                     "r2": r2_score(y.iloc[te_i], p.predict(X.iloc[te_i]))})

    return dict(X=X, y=y, X_te=X_te, y_te=y_te, ridge=ridge, alpha=alpha,
                feat_order=feat_order,
                r2_random=r2_score(y_te, ridge.predict(X_te)),
                loso=pd.DataFrame(loso))


def scenario(X, df, scm, days, sulph):
    r = {c: 0.0 for c in X.columns}
    r["exposure"] = float(days)
    r["sulphate"] = float(sulph)
    col = f"SCM_{scm}"
    if col in X.columns:
        r[col] = 1.0
    return pd.DataFrame([r])[X.columns]


# ==============================================================================
# FIGURES
# ==============================================================================

def fig1(df, fmt, outdir):
    """Dataset composition: observations per source, grouped by SCM category."""
    ct = pd.crosstab(df["scm_type"], df["source"])
    ct = ct.reindex(columns=[s for s in SOURCE_ORDER if s in ct.columns],
                    fill_value=0)
    ct = ct.loc[ct.sum(axis=1).sort_values(ascending=True).index]

    fig, ax = plt.subplots(figsize=(W2, 3.4))
    left = np.zeros(len(ct))
    for src in ct.columns:
        vals = ct[src].values
        ax.barh(ct.index, vals, left=left, height=0.68,
                color=SOURCE_FILL[src], hatch=SOURCE_HATCH[src],
                edgecolor="white", linewidth=0.6, label=src)
        left += vals

    for i, total in enumerate(ct.sum(axis=1).values):
        ax.text(total + 0.7, i, str(int(total)), va="center", fontsize=8)

    ax.set_xlabel("Number of observations")
    ax.set_ylabel("SCM category")
    ax.set_xlim(0, ct.sum(axis=1).max() * 1.13)
    ax.grid(axis="y", visible=False)
    ax.legend(frameon=False, loc="lower right")
    save(fig, "Fig1", fmt, outdir)


def fig2(m, fmt, outdir):
    """Predicted vs measured, random 20% test partition, with 1:1 line."""
    pred = m["ridge"].predict(m["X_te"])
    obs = m["y_te"].values

    fig, ax = plt.subplots(figsize=(W1 * 1.35, W1 * 1.35))
    lo = min(obs.min(), pred.min()) - 4
    hi = max(obs.max(), pred.max()) + 4
    ax.plot([lo, hi], [lo, hi], ls="--", color=C_LINE, lw=1.0, zorder=1)
    ax.scatter(obs, pred, s=34, facecolor=C_MID, edgecolor=C_DARK,
               linewidth=0.6, zorder=3)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_aspect("equal")
    ax.set_xlabel("Measured compressive strength (MPa)")
    ax.set_ylabel("Predicted compressive strength (MPa)")
    ax.legend(handles=[
        Line2D([], [], marker="o", ls="none", markerfacecolor=C_MID,
               markeredgecolor=C_DARK, markersize=5, label="Test observation"),
        Line2D([], [], ls="--", color=C_LINE, label="Perfect agreement")],
        frameon=False, loc="upper left")
    save(fig, "Fig2", fmt, outdir)


def fig3(m, fmt, outdir):
    """Standardised ridge coefficients, sorted, zero line marked."""
    coef = m["ridge"].named_steps["model"].coef_
    names = [n.replace("SCM_", "").replace("exposure", "Exposure duration")
              .replace("sulphate", "Sulphate concentration")
             for n in m["feat_order"]]
    s = pd.Series(coef, index=names).sort_values()

    fig, ax = plt.subplots(figsize=(W2, 3.6))
    colors = [C_NEG if v < 0 else C_POS for v in s.values]
    hatches = ["///" if v < 0 else "" for v in s.values]
    bars = ax.barh(s.index, s.values, height=0.68, color=colors,
                   edgecolor="black", linewidth=0.5)
    for b, h in zip(bars, hatches):
        b.set_hatch(h)
    ax.axvline(0, color="black", lw=0.8)

    pad = max(abs(s.min()), abs(s.max())) * 0.03
    for i, v in enumerate(s.values):
        ax.text(v + (pad if v >= 0 else -pad), i, f"{v:.2f}",
                va="center", ha="left" if v >= 0 else "right", fontsize=7.5)

    ax.set_xlabel("Standardised coefficient (MPa)")
    ax.set_xlim(s.min() * 1.22, s.max() * 1.28)
    ax.grid(axis="y", visible=False)
    save(fig, "Fig3", fmt, outdir)


def fig4(m, fmt, outdir):
    """R2 under random partitioning vs leave-one-source-out.

    Two panels sharing a category axis. A single linear axis cannot show
    0.968 and -157.92 together; a symlog axis compresses the difference the
    figure exists to convey. Panel (a) shows the near-zero region, panel (b)
    the full range, with the panel (a) window marked on panel (b).
    """
    loso = m["loso"].set_index("source").reindex(SOURCE_ORDER)
    labels = ["Random\nsplit"] + [f"LOSO\n{s}" for s in SOURCE_ORDER]
    vals = [m["r2_random"]] + list(loso["r2"].values)
    colors = [C_POS] + [C_NEG] * 3
    hatches = [""] + ["///"] * 3

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(W2, 4.6), sharex=True,
        gridspec_kw={"height_ratios": [1, 1.5], "hspace": 0.18})

    for ax in (ax1, ax2):
        bars = ax.bar(labels, vals, width=0.6, color=colors,
                      edgecolor="black", linewidth=0.5)
        for b, h in zip(bars, hatches):
            b.set_hatch(h)
        ax.axhline(0, color="black", lw=0.8)
        ax.grid(axis="x", visible=False)

    ax1.set_ylim(-20, 2)
    ax2.set_ylim(-172, 12)
    ax1.text(0.012, 0.90, "a", transform=ax1.transAxes, fontweight="bold", fontsize=10)
    ax2.text(0.012, 0.93, "b", transform=ax2.transAxes, fontweight="bold", fontsize=10)

    for i, v in enumerate(vals):
        if v > -20:
            ax1.text(i, v + (0.8 if v >= 0 else -1.9), f"{v:.3f}" if v > 0 else f"{v:.2f}",
                     ha="center", fontsize=8)
        ax2.text(i, v - 8 if v < 0 else v + 4, f"{v:.2f}" if v < 0 else f"{v:.3f}",
                 ha="center", fontsize=8)

    ax2.axhspan(-20, 2, color="black", alpha=0.06, zorder=0)
    ax2.annotate("range shown in (a)", xy=(3.42, -10), xytext=(3.42, -46),
                 fontsize=7, ha="center", color="0.35",
                 arrowprops=dict(arrowstyle="->", lw=0.6, color="0.5"))

    ax2.set_xlabel("Validation scheme", labelpad=6)
    fig.supylabel("Coefficient of determination, $R^2$", fontsize=9, x=0.005)
    save(fig, "Fig4", fmt, outdir)


def fig5(df, fmt, outdir):
    """Strength distribution by source, showing the unpopulated interval."""
    fig, (ax_h, ax_b) = plt.subplots(
        1, 2, figsize=(W2, 3.2), gridspec_kw={"width_ratios": [2, 1], "wspace": 0.28})

    bins = np.arange(0, 70, 2.5)
    for src in SOURCE_ORDER:
        s = df.loc[df["source"] == src, "strength"]
        if len(s):
            ax_h.hist(s, bins=bins, color=SOURCE_FILL[src], alpha=0.85,
                      hatch=SOURCE_HATCH[src], edgecolor="white",
                      linewidth=0.5, label=src)
    ax_h.axvspan(21.50, 43.00, color="0.5", alpha=0.16, zorder=0)
    ax_h.text(32.25, ax_h.get_ylim()[1] * 0.55, "no\nobservations",
              ha="center", va="center", fontsize=7.5, color="0.3")
    ax_h.set_xlabel("Compressive strength (MPa)")
    ax_h.set_ylabel("Number of observations")
    ax_h.legend(frameon=False, loc="upper right", handlelength=1.4)
    ax_h.text(0.015, 0.965, "a", transform=ax_h.transAxes,
              fontweight="bold", fontsize=10)

    data = [df.loc[df["source"] == s, "strength"].values for s in SOURCE_ORDER]
    # Updated 'labels' to 'tick_labels' to fix Matplotlib deprecation warning
    bp = ax_b.boxplot(data, tick_labels=SOURCE_ORDER, widths=0.55, patch_artist=True,
                      medianprops=dict(color="black", linewidth=1.1),
                      flierprops=dict(marker="o", markersize=3.2,
                                      markerfacecolor="white",
                                      markeredgecolor="black",
                                      markeredgewidth=0.6))
    for patch, src in zip(bp["boxes"], SOURCE_ORDER):
        patch.set_facecolor(SOURCE_FILL[src])
        patch.set_hatch(SOURCE_HATCH[src])
        patch.set_edgecolor("black")
        patch.set_linewidth(0.6)
    ax_b.axhspan(21.50, 43.00, color="0.5", alpha=0.16, zorder=0)
    ax_b.set_ylabel("Compressive strength (MPa)")
    ax_b.grid(axis="x", visible=False)
    ax_b.tick_params(axis="x", rotation=20)
    ax_b.text(0.03, 0.965, "b", transform=ax_b.transAxes,
              fontweight="bold", fontsize=10)

    save(fig, "Fig5", fmt, outdir)


def fig6(df, m, fmt, outdir):
    """CPA measured vs predicted; extrapolated segment dotted."""
    obs = df[(df["scm_type"] == F6_SCM) &
             np.isclose(df["sulphate"], F6_SULPHATE, atol=1e-3)]

    fig, ax = plt.subplots(figsize=(W2 * 0.86, 3.2))

    if len(obs):
        ax.scatter(obs["exposure"], obs["strength"], s=32, marker="o",
                   facecolor="white", edgecolor=C_DARK, linewidth=0.9,
                   zorder=3, label="Measured specimen")
        g = obs.groupby("exposure")["strength"].mean()
        ax.scatter(g.index, g.values, s=46, marker="D", color=C_DARK,
                   zorder=4, label="Measured mean")

    grid_in = np.linspace(obs["exposure"].min() if len(obs) else 7, F6_OBS_MAX, 200)
    ax.plot(grid_in, [m["ridge"].predict(
        scenario(m["X"], df, F6_SCM, d, F6_SULPHATE))[0] for d in grid_in],
        color=C_LINE, lw=1.4, zorder=2, label="Ridge prediction")

    grid_out = np.linspace(F6_OBS_MAX, F6_EXTRAP_MAX, 120)
    ax.plot(grid_out, [m["ridge"].predict(
        scenario(m["X"], df, F6_SCM, d, F6_SULPHATE))[0] for d in grid_out],
        color=C_LINE, lw=1.4, ls=":", zorder=2, label="Extrapolated")
    ax.axvline(F6_OBS_MAX, color="0.55", ls="--", lw=0.7)
    ax.text(F6_OBS_MAX + 3, ax.get_ylim()[0] + 0.6,
            "observed range ends", fontsize=7, color="0.35")

    ax.set_xlabel("Exposure duration (days)")
    ax.set_ylabel("Compressive strength (MPa)")
    ax.legend(frameon=False, loc="upper left", ncol=2, columnspacing=1.1)
    save(fig, "Fig6", fmt, outdir)


# ==============================================================================

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--outdir", default=str(OUTDIR))
    ap.add_argument("--format", default="tif", choices=["tif", "eps", "pdf", "png"],
                    help="tif (default) for submission; png for quick checks")
    # Simulate command-line arguments for Colab environment
    a = ap.parse_args(['--data', 'final_cleaned_sulfate_data_v2.csv'])

    set_style(a.format)
    outdir = Path(a.outdir)

    df = load(a.data)
    print(f"Loaded {len(df)} observations, {df['source'].nunique()} sources")

    m = fit_everything(df)
    print(f"alpha = {m['alpha']:.7g}   random-split R2 = {m['r2_random']:.4f}")
    for _, r in m["loso"].iterrows():
        print(f"  LOSO {r['source']}: R2 = {r['r2']:.2f}")
    print()

    fig1(df, a.format, outdir)
    fig2(m, a.format, outdir)
    fig3(m, a.format, outdir)
    fig4(m, a.format, outdir)
    fig5(df, a.format, outdir)
    fig6(df, m, a.format, outdir)

    print(f"\nDone. Six figures in {outdir.resolve()}")
    print("Check that the printed alpha and R2 above match Tables 3 and 5.")
    print("If Arial is not installed, matplotlib falls back to DejaVu Sans and")
    print("prints a warning. Install Arial or msttcorefonts before final export.")


if __name__ == "__main__":
    main()

Loaded 94 observations, 3 sources
alpha = 0.07149429   random-split R2 = 0.9676
  LOSO Source A: R2 = -157.92
  LOSO Source B: R2 = -1.83
  LOSO Source C: R2 = -14.55

  wrote figures/Fig1.tif
  wrote figures/Fig2.tif
  wrote figures/Fig3.tif
  wrote figures/Fig4.tif
  wrote figures/Fig5.tif
  wrote figures/Fig6.tif

Done. Six figures in /content/figures
Check that the printed alpha and R2 above match Tables 3 and 5.
If Arial is not installed, matplotlib falls back to DejaVu Sans and
prints a warning. Install Arial or msttcorefonts before final export.


In [ ]:
import shutil
from pathlib import Path
from google.colab import files

# Define the directory and zip filename
fig_dir = Path('figures')
zip_name = 'manuscript_figures.zip'

if fig_dir.exists() and fig_dir.is_dir():
    # Create the zip archive
    shutil.make_archive('manuscript_figures', 'zip', fig_dir)
    # Trigger the download
    files.download(zip_name)
    print(f'Successfully zipped {fig_dir} into {zip_name} and initiated download.')
else:
    print(f'Error: The directory {fig_dir} was not found. Please run the figure generation cell first.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Successfully zipped figures into manuscript_figures.zip and initiated download.
